<a href="https://colab.research.google.com/github/raki-rankawat/stm32-thesis/blob/main/Pipeline_Prune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
# =====================================================
# Install ONNX / ONNXRuntime Tools
# =====================================================
!pip -q install onnx onnxruntime onnxscript onnxruntime-tools

In [32]:
# =====================================================
# Imports
# =====================================================

import os
import time
import tarfile
import random
import shutil
import numpy as np
from pathlib import Path
from urllib.request import urlretrieve

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

import onnx
import onnxruntime as ort

from onnxruntime.quantization import (
    quantize_static,
    QuantType,
    QuantFormat,
    CalibrationDataReader
)

from google.colab import drive

In [33]:
# =====================================================
# Mount Google Drive
# =====================================================

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
# =====================================================
# Reproducibility + Device Setup
# =====================================================

SEED = 41

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [35]:
# =====================================================
# Dataset Configuration
# =====================================================

VWW_URL = "https://www.silabs.com/public/files/github/machine_learning/benchmarks/datasets/vw_coco2014_96.tar.gz"

BASE_DIR = Path("/content/vww_work")
ARCHIVE_PATH = BASE_DIR / "vw_coco2014_96.tar.gz"
EXTRACT_DIR = BASE_DIR / "extracted"

MANIFEST_BASE_DIR = Path("/content/drive/My Drive/vww_fixed_split_manifests")

N_PER_CLASS = 5000
VAL_RATIO = 0.20

IMG_SIZE = 96
EVAL_SAMPLES = 200
CALIB_SAMPLES = 200

In [36]:
# =====================================================
# File Names
# =====================================================

FP32_ONNX_NAME = "vww_mobilenetv2_fp32.onnx"
INT8_ONNX_NAME = "vww_mobilenetv2_int8_static_qdq.onnx"

CALIB_INPUT_NPZ = "vww_mobilenetv2_calib_train_200_input.npz"
VAL_INPUT_NPZ = "vww_mobilenetv2_val_200_input.npz"
VAL_LABELS_NPZ = "vww_mobilenetv2_val_200_labels.npz"
VAL_INPUT_LOGITS_NPZ = "vww_mobilenetv2_val_200_input_logits.npz"

EXPORT_DIR = Path("/content/drive/My Drive/Colab Notebooks/vww_mobilenetv2_prune_exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

In [37]:
# =====================================================
# Download VWW Dataset
# =====================================================

def download_vww():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    if ARCHIVE_PATH.exists() and ARCHIVE_PATH.stat().st_size > 0:
        print("✅ VWW archive already downloaded")
        return

    print("⬇️ Downloading VWW archive...")
    urlretrieve(VWW_URL, ARCHIVE_PATH)
    print("✅ Download complete:", ARCHIVE_PATH)

In [38]:
# =====================================================
# Safe Extract
# =====================================================

def safe_extract_tar(tar, path="."):
    path = Path(path).resolve()

    for member in tar.getmembers():
        member_path = (path / member.name).resolve()
        if not str(member_path).startswith(str(path)):
            raise RuntimeError("❌ Unsafe path detected in tar archive")

    tar.extractall(path)

In [39]:
# =====================================================
# Extract Dataset
# =====================================================

def extract_vww():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    if any(EXTRACT_DIR.iterdir()):
        print("✅ VWW already extracted")
        return

    print("📦 Extracting VWW archive...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        safe_extract_tar(tar, EXTRACT_DIR)

    print("✅ Extraction complete:", EXTRACT_DIR)

In [40]:
# =====================================================
# Locate Dataset Root
# =====================================================

def find_vww_root():
    for p in EXTRACT_DIR.rglob("person"):
        if p.is_dir() and (p.parent / "non_person").is_dir():
            return p.parent

    raise RuntimeError("❌ Could not find dataset directories")

In [41]:
# =====================================================
# Image Listing Helper
# =====================================================

def list_images(folder):
    exts = {".jpg", ".jpeg", ".png"}
    return sorted(
        [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in exts],
        key=lambda x: str(x)
    )

In [42]:
# =====================================================
# Manifest Helpers
# =====================================================

def manifest_paths():
    return {
        "train_person": MANIFEST_BASE_DIR / "train_person.txt",
        "val_person": MANIFEST_BASE_DIR / "val_person.txt",
        "train_non_person": MANIFEST_BASE_DIR / "train_non_person.txt",
        "val_non_person": MANIFEST_BASE_DIR / "val_non_person.txt",
    }

def manifests_ready():
    paths = manifest_paths()
    return all(p.exists() for p in paths.values())

def save_manifest(file_list, manifest_path):
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with open(manifest_path, "w") as f:
        for item in file_list:
            f.write(str(item) + "\n")

def load_manifest(manifest_path):
    with open(manifest_path, "r") as f:
        return [Path(line.strip()) for line in f if line.strip()]

In [43]:
# =====================================================
# Create Fixed Split Manifests Once
# =====================================================

def create_fixed_split_manifests(src_root):
    if manifests_ready():
        print("✅ Fixed split manifests already exist:", MANIFEST_BASE_DIR)
        return

    print("🧩 Creating fixed split manifests...")

    person_imgs = list_images(src_root / "person")
    nonperson_imgs = list_images(src_root / "non_person")

    if len(person_imgs) < N_PER_CLASS or len(nonperson_imgs) < N_PER_CLASS:
        raise ValueError(
            f"❌ Not enough images:\n"
            f"person: {len(person_imgs)} (need {N_PER_CLASS})\n"
            f"non_person: {len(nonperson_imgs)} (need {N_PER_CLASS})"
        )

    rng = random.Random(SEED)
    rng.shuffle(person_imgs)
    rng.shuffle(nonperson_imgs)

    person_sel = person_imgs[:N_PER_CLASS]
    nonperson_sel = nonperson_imgs[:N_PER_CLASS]

    def split_list(lst):
        n_val = int(len(lst) * VAL_RATIO)
        return lst[n_val:], lst[:n_val]   # train, val

    p_train, p_val = split_list(person_sel)
    n_train, n_val = split_list(nonperson_sel)

    paths = manifest_paths()
    save_manifest(p_train, paths["train_person"])
    save_manifest(p_val, paths["val_person"])
    save_manifest(n_train, paths["train_non_person"])
    save_manifest(n_val, paths["val_non_person"])

    print("✅ Fixed split manifests saved at:", MANIFEST_BASE_DIR)

In [44]:
# =====================================================
# Custom Dataset from Manifest
# =====================================================

class VWWManifestDataset(Dataset):
    def __init__(self, person_manifest, nonperson_manifest, transform=None):
        self.transform = transform
        self.samples = []

        person_files = load_manifest(person_manifest)
        nonperson_files = load_manifest(nonperson_manifest)

        for p in nonperson_files:
            self.samples.append((p, 0))

        for p in person_files:
            self.samples.append((p, 1))

        self.class_to_idx = {"non_person": 0, "person": 1}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [45]:
# =====================================================
# Prepare Dataset
# =====================================================

print("Step 1/4: Download")
download_vww()

print("Step 2/4: Extract")
extract_vww()

print("Step 3/4: Find root")
vww_root = find_vww_root()
print("✅ Dataset root:", vww_root)

print("Step 4/4: Create fixed manifests")
create_fixed_split_manifests(vww_root)

Step 1/4: Download
✅ VWW archive already downloaded
Step 2/4: Extract
✅ VWW already extracted
Step 3/4: Find root
✅ Dataset root: /content/vww_work/extracted/vw_coco2014_96
Step 4/4: Create fixed manifests
✅ Fixed split manifests already exist: /content/drive/My Drive/vww_fixed_split_manifests


In [46]:
# =====================================================
# Evaluation / Calibration Dataset
# =====================================================

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    )
])

paths = manifest_paths()

val_dataset = VWWManifestDataset(
    person_manifest=paths["val_person"],
    nonperson_manifest=paths["val_non_person"],
    transform=eval_transform
)

train_dataset_for_calib = VWWManifestDataset(
    person_manifest=paths["train_person"],
    nonperson_manifest=paths["train_non_person"],
    transform=eval_transform
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

calib_loader = DataLoader(
    train_dataset_for_calib,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("Class mapping:", val_dataset.class_to_idx)
print("Validation samples:", len(val_dataset))
print("Training samples for calibration:", len(train_dataset_for_calib))

Class mapping: {'non_person': 0, 'person': 1}
Validation samples: 2000
Training samples for calibration: 8000


In [47]:
# =====================================================
# MobileNetV2 Block
# =====================================================

class InvertedResidual(nn.Module):
    def __init__(self, in_channels, out_channels, stride, expand_ratio):
        super().__init__()

        hidden_dim = in_channels * expand_ratio
        self.use_residual = stride == 1 and in_channels == out_channels

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),

            nn.Conv2d(
                hidden_dim,
                hidden_dim,
                3,
                stride=stride,
                padding=1,
                groups=hidden_dim,
                bias=False
            ),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),

            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        if self.use_residual:
            return x + self.block(x)
        return self.block(x)

In [48]:
# =====================================================
# MobileNetV2 Model
# =====================================================

class VWW_MobileNetV2(nn.Module):
    def __init__(self):
        super().__init__()

        self.initial = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True)
        )

        self.features = nn.Sequential(
            InvertedResidual(32, 16, 1, 1),

            InvertedResidual(16, 24, 2, 6),
            InvertedResidual(24, 24, 1, 6),

            InvertedResidual(24, 32, 2, 6),
            InvertedResidual(32, 32, 1, 6),

            InvertedResidual(32, 64, 2, 6),
            InvertedResidual(64, 64, 1, 6),
        )

        self.final_conv = nn.Sequential(
            nn.Conv2d(64, 512, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU6(inplace=True)
        )

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(512, 2)

        self._initialize_weights()

    def forward(self, x):
        x = self.initial(x)
        x = self.features(x)
        x = self.final_conv(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

In [49]:
# =====================================================
# Load Trained MobileNet Weights
# =====================================================

checkpoint_path = "/content/drive/My Drive/Colab Notebooks/mobilenetv2_pruned_unst_30pct.pth"

model = VWW_MobileNetV2().to(device)
state_dict = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print("✅ Loaded:", checkpoint_path)
print("✅ Model device:", next(model.parameters()).device)

✅ Loaded: /content/drive/My Drive/Colab Notebooks/mobilenetv2_pruned_unst_30pct.pth
✅ Model device: cpu


In [50]:
# =====================================================
# PyTorch Accuracy on First N Validation Samples
# =====================================================

correct = 0
total = 0

with torch.no_grad():
    for i, (x, y) in enumerate(val_loader):
        if i >= EVAL_SAMPLES:
            break

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        out = model(x)
        pred = out.argmax(dim=1)

        correct += (pred == y).sum().item()
        total += y.size(0)

print(f"PyTorch accuracy on first {EVAL_SAMPLES} validation samples: {100 * correct / total:.2f}%")

PyTorch accuracy on first 200 validation samples: 81.00%


In [51]:
# =====================================================
# Export FP32 ONNX
# =====================================================

def export_onnx(model, onnx_path):
    model.eval()
    dummy = torch.randn(1, 3, 96, 96, device=device)

    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        input_names=["input"],
        output_names=["logits"],
        export_params=True,
        opset_version=18,
        do_constant_folding=True,
        dynamic_axes={
            "input": {0: "batch_size"},
            "logits": {0: "batch_size"}
        },
        dynamo=False
    )

    onnx.checker.check_model(onnx_path, full_check=False)
    print(f"✅ ONNX model saved to: {onnx_path}")

export_onnx(model, FP32_ONNX_NAME)

/tmp/ipykernel_1191/1410384257.py:9: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


✅ ONNX model saved to: vww_mobilenetv2_fp32.onnx


In [52]:
# =====================================================
# Save Validation Input + Labels + Debug Logits
# - VAL_INPUT_NPZ: used on STM32 for FP32 and INT8
# - VAL_LABELS_NPZ: ground truth labels
# - VAL_INPUT_LOGITS_NPZ: optional debug file
# =====================================================

def save_validation_files(model, loader, n_samples=200):
    inputs_nchw = []
    logits = []
    labels = []

    model.eval()

    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= n_samples:
                break

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            out = model(x)

            x_nchw = x.detach().cpu().numpy().astype(np.float32)[0]       # (3,96,96)
            out_np = out.detach().cpu().numpy().astype(np.float32)[0]     # (2,)

            inputs_nchw.append(x_nchw)
            logits.append(out_np)
            labels.append(int(y.item()))

    inputs_nchw = np.stack(inputs_nchw, axis=0)   # (N,3,96,96)
    logits = np.stack(logits, axis=0)             # (N,2)
    labels = np.array(labels, dtype=np.int32)     # (N,)

    np.savez(VAL_INPUT_NPZ, input=inputs_nchw)
    np.savez(VAL_LABELS_NPZ, label=labels)
    np.savez(VAL_INPUT_LOGITS_NPZ, input=inputs_nchw, logits=logits)

    print("✅ Saved validation input:", VAL_INPUT_NPZ)
    print("✅ Saved validation labels:", VAL_LABELS_NPZ)
    print("✅ Saved validation debug file:", VAL_INPUT_LOGITS_NPZ)
    print("Input shape:", inputs_nchw.shape)
    print("Logits shape:", logits.shape)
    print("Labels shape:", labels.shape)
    print("Input min/max:", inputs_nchw.min(), inputs_nchw.max())

save_validation_files(model, val_loader, n_samples=EVAL_SAMPLES)

✅ Saved validation input: vww_mobilenetv2_val_200_input.npz
✅ Saved validation labels: vww_mobilenetv2_val_200_labels.npz
✅ Saved validation debug file: vww_mobilenetv2_val_200_input_logits.npz
Input shape: (200, 3, 96, 96)
Logits shape: (200, 2)
Labels shape: (200,)
Input min/max: -2.117904 2.64


In [53]:
# =====================================================
# Save Calibration Input NPZ
# - TRAIN split only
# - input only
# =====================================================

def make_input_npz(dataset, N=200, out_path="samples_input.npz"):
    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    xs = []

    with torch.no_grad():
        for i, (x, _) in enumerate(loader):
            if i >= N:
                break

            x_np = x.numpy().astype(np.float32)[0]   # (3,96,96)
            xs.append(x_np)

    xs = np.stack(xs, axis=0)
    np.savez(out_path, input=xs)

    print("✅ Saved input file:", out_path, xs.shape)
    return out_path

calib_npz = make_input_npz(
    train_dataset_for_calib,
    N=CALIB_SAMPLES,
    out_path=CALIB_INPUT_NPZ
)

✅ Saved input file: vww_mobilenetv2_calib_train_200_input.npz (200, 3, 96, 96)


In [54]:
# =====================================================
# Calibration Reader
# =====================================================

class CalibReader(CalibrationDataReader):
    def __init__(self, npz_path, input_name="input"):
        self.x = np.load(npz_path)["input"].astype(np.float32)
        self.input_name = input_name
        self.i = 0

    def get_next(self):
        if self.i >= len(self.x):
            return None

        batch = self.x[self.i:self.i + 1]   # (1,3,96,96)
        self.i += 1
        return {self.input_name: batch}

    def rewind(self):
        self.i = 0

In [55]:
# =====================================================
# Export INT8 QDQ ONNX
# =====================================================

def quantize_int8_qdq(
    fp32_onnx=FP32_ONNX_NAME,
    calib_npz=CALIB_INPUT_NPZ,
    int8_onnx=INT8_ONNX_NAME
):
    reader = CalibReader(calib_npz, input_name="input")

    quantize_static(
        model_input=fp32_onnx,
        model_output=int8_onnx,
        calibration_data_reader=reader,
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QInt8,
        weight_type=QuantType.QInt8,
        per_channel=True,
    )

    print("✅ Saved INT8:", int8_onnx)
    return int8_onnx

quantize_int8_qdq(
    fp32_onnx=FP32_ONNX_NAME,
    calib_npz=CALIB_INPUT_NPZ,
    int8_onnx=INT8_ONNX_NAME
)

✅ Saved INT8: vww_mobilenetv2_int8_static_qdq.onnx


'vww_mobilenetv2_int8_static_qdq.onnx'

In [56]:
# =====================================================
# Optional: ONNX Runtime FP32 Check
# =====================================================

session = ort.InferenceSession(
    FP32_ONNX_NAME,
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

sample_input = np.load(VAL_INPUT_NPZ)["input"][0:1].astype(np.float32)
ort_out = session.run([output_name], {input_name: sample_input})[0]

print("✅ ONNX Runtime output shape:", ort_out.shape)
print("Sample logits:", ort_out[0])

✅ ONNX Runtime output shape: (1, 2)
Sample logits: [-0.2210146  0.2635864]


In [57]:
# =====================================================
# Copy Export Files to Drive
# =====================================================

files_to_copy = [
    FP32_ONNX_NAME,
    INT8_ONNX_NAME,
    CALIB_INPUT_NPZ,
    VAL_INPUT_NPZ,
    VAL_LABELS_NPZ,
    VAL_INPUT_LOGITS_NPZ,
]

for file_name in files_to_copy:
    src = Path(file_name)
    if src.exists():
        shutil.copy2(src, EXPORT_DIR / src.name)
        print(f"✅ Copied {src.name}")

print("✅ Export folder:", EXPORT_DIR)

✅ Copied vww_mobilenetv2_fp32.onnx
✅ Copied vww_mobilenetv2_int8_static_qdq.onnx
✅ Copied vww_mobilenetv2_calib_train_200_input.npz
✅ Copied vww_mobilenetv2_val_200_input.npz
✅ Copied vww_mobilenetv2_val_200_labels.npz
✅ Copied vww_mobilenetv2_val_200_input_logits.npz
✅ Export folder: /content/drive/My Drive/Colab Notebooks/vww_mobilenetv2_prune_exports


In [63]:
# =====================================================
# Compute Accuracy from STM32 Output NPZ
# Use this after STM32 generates:
# - network_val_200_outputs_fp32.npz
# - network_val_200_outputs_int8.npz
# =====================================================

def compute_accuracy(
    labels_npz_path,
    outputs_npz_path,
    output_key="c_outputs_1",
    num_classes=2,
    as_percentage=False
):
    labels = np.load(labels_npz_path)["label"].astype(np.int64)
    out = np.load(outputs_npz_path)

    raw = out[output_key]
    total_values = raw.size
    expected_values = len(labels) * num_classes

    print("Labels:", len(labels))
    print("Output raw shape:", raw.shape)
    print("Output raw size:", total_values)
    print("Expected size:", expected_values)

    if total_values != expected_values:
        raise ValueError(
            f"Size mismatch: output has {total_values} values, "
            f"but expected {expected_values} = {len(labels)} x {num_classes}. "
            f"Use matching input/label/output files from the same run."
        )

    logits = raw.reshape(len(labels), num_classes)
    pred = np.argmax(logits, axis=1)

    acc = (pred == labels).mean()
    return acc * 100 if as_percentage else acc

In [62]:
# =====================================================
# HOW TO USE ON STM32
# =====================================================
#
# FP32 STM32 run:
#   model    -> vww_mobilenetv2_fp32.onnx
#   valinput -> vww_mobilenetv2_val_200_input.npz
#
# INT8 STM32 run:
#   model    -> vww_mobilenetv2_int8_static_qdq.onnx
#   valinput -> vww_mobilenetv2_val_200_input.npz
#
# DO NOT use:
#   vww_mobilenetv2_calib_train_200_input.npz
# for final STM32 accuracy testing.
#
# After each STM32 run, rename the generated output file:
#   network_val_io.npz -> network_val_200_outputs_fp32.npz
#   network_val_io.npz -> network_val_200_outputs_int8.npz
#
# =====================================================

In [64]:
# =====================================================
# EXAMPLE: FP32 Accuracy
# Uncomment after FP32 STM32 run and after renaming file
# =====================================================

fp32_acc = compute_accuracy(
    VAL_LABELS_NPZ,
    "network_val_200_outputs_fp32.npz",
    output_key="c_outputs_1",
    num_classes=2,
    as_percentage=True
)

print("FP32 STM32 accuracy:", fp32_acc)

Labels: 200
Output raw shape: (200, 1, 1, 2)
Output raw size: 400
Expected size: 400
FP32 STM32 accuracy: 81.0


In [65]:
# =====================================================
# EXAMPLE: INT8 Accuracy
# Uncomment after INT8 STM32 run and after renaming file
# =====================================================

int8_acc = compute_accuracy(
    VAL_LABELS_NPZ,
    "network_val_200_outputs_int8.npz",
    output_key="c_outputs_1",
    num_classes=2,
    as_percentage=True
)

print("INT8 STM32 accuracy:", int8_acc)

Labels: 200
Output raw shape: (200, 1, 1, 2)
Output raw size: 400
Expected size: 400
INT8 STM32 accuracy: 79.5
